# Neural Network — Supervised Loan Default Prediction

This notebook builds and evaluates a Multi-Layer Perceptron (MLP) neural network for binary classification of loan default. It covers:

1. Temporal train / validation / test split  
2. Class imbalance strategy (`pos_weight` in `BCEWithLogitsLoss`)  
3. MLP architecture with **BatchNorm + Dropout**  
4. Hyperparameter search (learning rate, hidden dims, dropout)  
5. Early stopping and learning-rate scheduling  
6. Evaluation: **ROC-AUC · PR-AUC · KS Statistic · F1 · Confusion Matrix**  
7. Threshold optimization for a credit-risk business context  
8. Permutation feature importance  
9. Interpretation and justification of every choice  

**Input:** `data/features.parquet`, `data/temporal_split_index.parquet`  
**Output:** `models/nn_model.pt`, `models/nn_results.csv`

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    classification_report, confusion_matrix,
    roc_curve, precision_recall_curve,
)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

# Reproducibility 
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device  : {device}")
print(f"PyTorch : {torch.__version__}")

os.makedirs('models', exist_ok=True)

Device  : cpu
PyTorch : 2.11.0


## 1. Load Data & Temporal Split

### Why temporal — not random — splitting?

**Split strategy:**  
- **Train** — loans issued before 2016-01-01 (≈ 70 %)  
- **Validation** — 2016 to 2017 (≈ 15 %) — *hyperparameter selection only*  
- **Test** — 2017-01-01 onward (≈ 15 %) — *held out until final evaluation*

## 2. Class Imbalance Strategy

The dataset has ≈ 20 % default rate (≈ 1:4 class ratio). We handle this with
**`pos_weight`** inside `BCEWithLogitsLoss`:
With 1 M+ rows, `pos_weight` is the practical and principled choice.

In [ ]:
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
pos_weight_value = n_neg / n_pos

print(f"Fully Paid  (0): {n_neg:,}")
print(f"Charged Off (1): {n_pos:,}")
print(f"Imbalance ratio: {n_neg/n_pos:.2f}:1")
print(f"pos_weight      : {pos_weight_value:.4f}")

# Visualize
fig, ax = plt.subplots(figsize=(7, 4))
vals   = [n_neg, n_pos]
labels = ['Fully Paid (0)', 'Charged Off (1)']
colors = ['#2ecc71', '#e74c3c']
ax.bar(labels, vals, color=colors, edgecolor='black', linewidth=0.5)
for i, v in enumerate(vals):
    ax.text(i, v + 8_000, f'{v:,}\n({v/len(y_train)*100:.1f}%)',
            ha='center', fontsize=11)
ax.set_title('Class Distribution — Training Set', fontsize=14)
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 3. PyTorch DataLoaders

In [ ]:
BATCH_SIZE = 1024

def make_loader(X: np.ndarray, y: np.ndarray,
                shuffle: bool = True, batch_size: int = BATCH_SIZE) -> DataLoader:
    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32),
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      num_workers=0, pin_memory=(device.type == 'cuda'))

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader   = make_loader(X_val,   y_val,   shuffle=False)
test_loader  = make_loader(X_test,  y_test,  shuffle=False)

print(f"Train batches : {len(train_loader):,}")
print(f"Val batches   : {len(val_loader):,}")
print(f"Test batches  : {len(test_loader):,}")

## 4. Model Architecture


In [ ]:
class LoanDefaultMLP(nn.Module):
    """
    Multi-Layer Perceptron for binary loan default classification.

    Architecture: [Linear → BatchNorm1d → ReLU → Dropout] × N → Linear(1)
    Output is a raw logit (sigmoid applied externally or via BCEWithLogitsLoss).
    """
    def __init__(self, input_dim: int, hidden_dims: list[int], dropout_rate: float = 0.3):
        super().__init__()
        layers: list[nn.Module] = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout_rate)]
            prev = h
        layers.append(nn.Linear(prev, 1))   # output logit
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(1)

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


#Preview default architecture
INPUT_DIM = X_train.shape[1]
_demo = LoanDefaultMLP(INPUT_DIM, hidden_dims=[256, 128, 64], dropout_rate=0.3)
print(_demo)
print(f"\nTrainable parameters: {_demo.count_parameters():,}")

## 5. Training Utilities
**Why ROC-AUC (not loss) for early stopping?**  
- Loss is a proxy; AUC directly measures the rank-ordering quality we care about  
- AUC is threshold-independent, so it captures model quality across the full operating range  
- PR-AUC is also reported (more informative under imbalance) but ROC-AUC is the primary signal

In [ ]:
# Training loop 
def train_epoch(model, loader, criterion, optimizer, device) -> float:
    model.train()
    total_loss = 0.0
    for Xb, yb in loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(Xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(yb)
    return total_loss / len(loader.dataset)


# Evaluation 
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_logits, all_labels = 0.0, [], []
    with torch.no_grad():
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            logits = model(Xb)
            total_loss += criterion(logits, yb).item() * len(yb)
            all_logits.append(logits.cpu())
            all_labels.append(yb.cpu())
    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()
    probs  = torch.sigmoid(torch.tensor(logits)).numpy()
    return (total_loss / len(loader.dataset),
            roc_auc_score(labels, probs),
            average_precision_score(labels, probs),
            probs, labels)


# Early stopping 
class EarlyStopping:
    """Save best model weights; stop if metric does not improve for `patience` epochs."""
    def __init__(self, patience: int = 7, min_delta: float = 1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = None
        self.counter    = 0
        self.best_state = None

    def step(self, score: float, model: nn.Module) -> bool:
        """Returns True if training should stop."""
        improved = (self.best_score is None or
                    score > self.best_score + self.min_delta)
        if improved:
            self.best_score = score
            self.counter    = 0
            self.best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
        return self.counter >= self.patience

    def restore(self, model: nn.Module) -> None:
        if self.best_state:
            model.load_state_dict(self.best_state)


# Full training run 
def train_model(cfg, X_tr, y_tr, X_vl, y_vl,
                device, pw_value, max_epochs=50, patience=7, verbose=True):
    tr_loader = make_loader(X_tr, y_tr, shuffle=True,  batch_size=cfg['batch_size'])
    vl_loader = make_loader(X_vl, y_vl, shuffle=False, batch_size=cfg['batch_size'])

    model     = LoanDefaultMLP(X_tr.shape[1], cfg['hidden_dims'], cfg['dropout_rate']).to(device)
    criterion = nn.BCEWithLogitsLoss(
                    pos_weight=torch.tensor([pw_value], dtype=torch.float32).to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg['lr'],
                                 weight_decay=cfg.get('weight_decay', 1e-5))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, mode='max', factor=0.5, patience=3, verbose=False)
    stopper   = EarlyStopping(patience=patience)
    history   = dict(train_loss=[], val_loss=[], val_roc=[], val_pr=[])

    for epoch in range(1, max_epochs + 1):
        tr_loss = train_epoch(model, tr_loader, criterion, optimizer, device)
        vl_loss, vl_roc, vl_pr, _, _ = evaluate(model, vl_loader, criterion, device)
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['val_roc'].append(vl_roc)
        history['val_pr'].append(vl_pr)
        scheduler.step(vl_roc)

        if verbose and (epoch % 5 == 0 or epoch == 1):
            print(f"  Ep {epoch:3d}:  train_loss={tr_loss:.4f}  "
                  f"val_loss={vl_loss:.4f}  ROC-AUC={vl_roc:.4f}  PR-AUC={vl_pr:.4f}")

        if stopper.step(vl_roc, model):
            if verbose:
                print(f"  ✓ Early stop @ epoch {epoch}  (best ROC-AUC={stopper.best_score:.4f})")
            break

    stopper.restore(model)
    return model, history, stopper.best_score, len(history['val_roc'])

print("Training utilities ready.")

## 6. Hyperparameter Search

### Search space
Each config is trained for up to **30 epochs** with **early stopping (patience=5)** monitored on **validation ROC-AUC**.  

In [ ]:
SEARCH_CONFIGS = [
    {'name': 'small_low_drop',   'hidden_dims': [128, 64],       'dropout_rate': 0.2, 'lr': 1e-3,  'batch_size': 1024},
    {'name': 'medium_med_drop',  'hidden_dims': [256, 128, 64],  'dropout_rate': 0.3, 'lr': 1e-3,  'batch_size': 1024},
    {'name': 'medium_low_lr',    'hidden_dims': [256, 128, 64],  'dropout_rate': 0.3, 'lr': 3e-4,  'batch_size': 1024},
    {'name': 'large_high_drop',  'hidden_dims': [512, 256, 128], 'dropout_rate': 0.4, 'lr': 1e-3,  'batch_size': 1024},
    {'name': 'large_low_lr',     'hidden_dims': [512, 256, 128], 'dropout_rate': 0.3, 'lr': 3e-4,  'batch_size': 1024},
]

search_results = []
print("=" * 72)
print("HYPERPARAMETER SEARCH  (val metric = ROC-AUC, max_epochs=30, patience=5)")
print("=" * 72)

for cfg in SEARCH_CONFIGS:
    print(f"\n▶  {cfg['name']}")
    print(f"   hidden={cfg['hidden_dims']}  dropout={cfg['dropout_rate']}  lr={cfg['lr']}")
    _, history, best_auc, n_ep = train_model(
        cfg, X_train, y_train, X_val, y_val,
        device, pos_weight_value, max_epochs=30, patience=5, verbose=True,
    )
    search_results.append({'name': cfg['name'],
                           'hidden_dims': str(cfg['hidden_dims']),
                           'dropout_rate': cfg['dropout_rate'],
                           'lr': cfg['lr'],
                           'best_val_roc_auc': round(best_auc, 4),
                           'epochs_run': n_ep})
    print(f"   → Best Val ROC-AUC: {best_auc:.4f}")

results_df = pd.DataFrame(search_results).sort_values('best_val_roc_auc', ascending=False)
print("\n" + "=" * 72)
print("SEARCH SUMMARY (sorted by Val ROC-AUC)")
print("=" * 72)
print(results_df.to_string(index=False))

In [ ]:
best_name   = results_df.iloc[0]['name']
best_config = next(c for c in SEARCH_CONFIGS if c['name'] == best_name)

print(f"✓ Best config: {best_name}")
print(f"  hidden_dims  : {best_config['hidden_dims']}")
print(f"  dropout_rate : {best_config['dropout_rate']}")
print(f"  lr           : {best_config['lr']}")

# Viz
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#e74c3c' if n == best_name else '#3498db' for n in results_df['name']]
bars = ax.barh(results_df['name'], results_df['best_val_roc_auc'],
               color=colors, edgecolor='white')
ax.set_xlabel('Best Validation ROC-AUC', fontsize=12)
ax.set_title('Hyperparameter Search Results', fontsize=14)
low = results_df['best_val_roc_auc'].min() - 0.01
ax.set_xlim([low, results_df['best_val_roc_auc'].max() + 0.01])
for bar, val in zip(bars, results_df['best_val_roc_auc']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)
ax.legend(handles=[
    plt.Rectangle((0,0),1,1, color='#e74c3c', label='Best'),
    plt.Rectangle((0,0),1,1, color='#3498db', label='Others'),
], loc='lower right')
plt.tight_layout()
plt.show()

## 7. Final Model Training


In [ ]:
X_tv = np.vstack([X_train, X_val])
y_tv = np.concatenate([y_train, y_val])
pw_tv = (y_tv == 0).sum() / (y_tv == 1).sum()

print(f"Train+Val size : {X_tv.shape[0]:,}")
print(f"Test size      : {X_test.shape[0]:,}")
print(f"pos_weight (tv): {pw_tv:.4f}")
print(f"\nRetraining: {best_name}")
print("=" * 60)

final_model, final_history, _, _ = train_model(
    best_config, X_tv, y_tv, X_test, y_test,
    device, pw_tv, max_epochs=60, patience=10, verbose=True,
)

torch.save({
    'model_state_dict': final_model.state_dict(),
    'config':           best_config,
    'input_dim':        X_tv.shape[1],
    'feature_names':    list(X.columns),
}, 'models/nn_model.pt')
print("\n✓ Model saved → models/nn_model.pt")

In [ ]:
# Training curves
epochs_x = range(1, len(final_history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_x, final_history['train_loss'], label='Train Loss', color='#3498db', lw=2)
axes[0].plot(epochs_x, final_history['val_loss'],   label='Test Loss',  color='#e74c3c', lw=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE Loss')
axes[0].set_title('Loss Curves', fontsize=13); axes[0].legend()

axes[1].plot(epochs_x, final_history['val_roc'], label='Test ROC-AUC', color='#2ecc71', lw=2)
axes[1].plot(epochs_x, final_history['val_pr'],  label='Test PR-AUC',  color='#e67e22', lw=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('AUC')
axes[1].set_title('Test AUC Over Training', fontsize=13)
axes[1].legend(); axes[1].set_ylim([0, 1])

plt.suptitle('Final Model Training Curves', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

## 8. Model Evaluation

In [ ]:
# test predictions 
criterion_eval = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([pw_tv], dtype=torch.float32).to(device))

test_loss, test_roc, test_pr, test_probs, test_labels = evaluate(
    final_model, test_loader, criterion_eval, device)

# KS Statistic 
fpr_arr, tpr_arr, thresholds_roc = roc_curve(test_labels, test_probs)
ks_arr  = tpr_arr - fpr_arr
ks_stat = ks_arr.max()
ks_idx  = ks_arr.argmax()
ks_thresh = thresholds_roc[ks_idx]

# F1 @ 0.5 
preds_05 = (test_probs >= 0.5).astype(int)
f1_05    = f1_score(test_labels, preds_05)

print("=" * 55)
print("FINAL MODEL — TEST SET METRICS")
print("=" * 55)
print(f"  ROC-AUC          : {test_roc:.4f}")
print(f"  PR-AUC           : {test_pr:.4f}")
print(f"  KS Statistic     : {ks_stat:.4f}  (threshold={ks_thresh:.3f})")
print(f"  F1 (t=0.50)      : {f1_05:.4f}")
print(f"  BCE Loss         : {test_loss:.4f}")
print("=" * 55)

print("\nClassification Report (threshold = 0.5):")
print(classification_report(test_labels, preds_05,
                             target_names=['Fully Paid', 'Charged Off']))

In [ ]:
prec_arr, rec_arr, _ = precision_recall_curve(test_labels, test_probs)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ROC Curve 
axes[0].plot(fpr_arr, tpr_arr, color='#3498db', lw=2.5,
             label=f'Neural Network  AUC={test_roc:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
axes[0].scatter(fpr_arr[ks_idx], tpr_arr[ks_idx],
                color='red', s=100, zorder=5,
                label=f'KS={ks_stat:.3f}  (t={ks_thresh:.3f})')
axes[0].fill_between(fpr_arr, tpr_arr, alpha=0.08, color='#3498db')
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate',  fontsize=12)
axes[0].set_title('ROC Curve', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# PR Curve
baseline = test_labels.mean()
axes[1].axhline(baseline, color='k', linestyle='--', lw=1,
                label=f'Random  precision={baseline:.2f}')
axes[1].plot(rec_arr, prec_arr, color='#e74c3c', lw=2.5,
             label=f'Neural Network  AP={test_pr:.3f}')
axes[1].fill_between(rec_arr, prec_arr, alpha=0.08, color='#e74c3c')
axes[1].set_xlabel('Recall',    fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title('Precision-Recall Curve', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Neural Network — Test Set Performance', fontsize=15, y=1.01)
plt.tight_layout(); plt.show()

## 9. Confusion Matrix & Threshold Optimization

In [ ]:
thresholds = np.arange(0.10, 0.90, 0.025)
rows = []
for t in thresholds:
    preds = (test_probs >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(test_labels, preds).ravel()
    rows.append({'threshold': round(float(t), 3),
                 'precision': round(tp / (tp + fp + 1e-9), 4),
                 'recall':    round(tp / (tp + fn + 1e-9), 4),
                 'f1':        round(f1_score(test_labels, preds, zero_division=0), 4),
                 'fpr':       round(fp / (fp + tn + 1e-9), 4)})

thresh_df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(thresh_df['threshold'], thresh_df['precision'], label='Precision', color='#3498db', lw=2)
axes[0].plot(thresh_df['threshold'], thresh_df['recall'],    label='Recall',    color='#e74c3c', lw=2)
axes[0].plot(thresh_df['threshold'], thresh_df['f1'],        label='F1',        color='#2ecc71', lw=2, ls='--')
axes[0].axvline(0.5,      color='gray',   ls=':',  label='t=0.50 (default)')
axes[0].axvline(ks_thresh, color='purple', ls=':',  label=f't={ks_thresh:.2f} (KS)')
axes[0].set_xlabel('Threshold'); axes[0].set_ylabel('Score')
axes[0].set_title('Precision / Recall / F1 vs Threshold', fontsize=13)
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(thresh_df['threshold'], thresh_df['fpr'],    color='#e67e22', lw=2, label='FPR (good loans rejected)')
axes[1].plot(thresh_df['threshold'], thresh_df['recall'], color='#e74c3c', lw=2, label='Recall (defaults caught)')
axes[1].axvline(0.5, color='gray', ls=':', label='t=0.50')
axes[1].set_xlabel('Threshold')
axes[1].set_title('Business Tradeoff: Recall vs FPR', fontsize=13)
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
key_t = sorted(set([0.30, 0.40, 0.50, round(ks_thresh, 2), 0.60, 0.70]))
print("\nKey Operating Points:")
print(thresh_df[thresh_df['threshold'].isin(key_t)].to_string(index=False))

In [ ]:
# Confusion matrices at t=0.50 and t=KS 
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (t, label) in zip(axes, [
        (0.5,       'Default Threshold  (t=0.50)'),
        (ks_thresh, f'KS Optimal  (t={ks_thresh:.2f})')]):
    preds = (test_probs >= t).astype(int)
    cm    = confusion_matrix(test_labels, preds)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    annot = np.array([[f'{v:,}\n({p:.1%})' for v, p in zip(r, pr)]
                      for r, pr in zip(cm, cm_pct)])
    sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', ax=ax, cbar=False,
                xticklabels=['Pred: Paid', 'Pred: Default'],
                yticklabels=['True: Paid', 'True: Default'])
    ax.set_title(label, fontsize=12)
    ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')

plt.tight_layout(); plt.show()

# Business summary at t=0.5
tn, fp, fn, tp = confusion_matrix(test_labels, preds_05).ravel()
print("Business Interpretation (t=0.5):")
print(f"  ✓ Defaults correctly flagged  : {tp:>8,}")
print(f"  ✓ Good loans correctly approved: {tn:>8,}")
print(f"  ✗ Good loans wrongly rejected  : {fp:>8,}   ← lender loses revenue")
print(f"  ✗ Defaults missed (risky!)     : {fn:>8,}   ← lender loses principal")
print(f"  Recall on defaults : {tp/(tp+fn):.1%}")
print(f"  FPR on good loans  : {fp/(fp+tn):.1%}")

## 10. Permutation Feature Importance

Neural networks lack built-in feature importance. We use **permutation importance**:

> Randomly shuffle one feature column → measure AUC drop → larger drop = more important

In [ ]:
def permutation_importance(model, X_te, y_te, feature_names, device,
                            n_repeats: int = 3, top_k: int = 25):
    model.eval()
    Xt = torch.tensor(X_te, dtype=torch.float32).to(device)
    with torch.no_grad():
        base_probs = torch.sigmoid(model(Xt)).cpu().numpy()
    base_auc = roc_auc_score(y_te, base_probs)

    records = []
    for i, feat in enumerate(feature_names):
        drops = []
        for _ in range(n_repeats):
            X_perm = X_te.copy()
            rng = np.random.default_rng()
            rng.shuffle(X_perm[:, i])
            Xp = torch.tensor(X_perm, dtype=torch.float32).to(device)
            with torch.no_grad():
                perm_probs = torch.sigmoid(model(Xp)).cpu().numpy()
            drops.append(base_auc - roc_auc_score(y_te, perm_probs))
        records.append({'feature': feat,
                        'importance': float(np.mean(drops)),
                        'std': float(np.std(drops))})

    imp_df = (pd.DataFrame(records)
                .sort_values('importance', ascending=False)
                .head(top_k)
                .reset_index(drop=True))
    return imp_df, base_auc


print("Computing permutation importance — may take 1-2 minutes …")
imp_df, base_auc = permutation_importance(
    final_model, X_test, test_labels, list(X.columns), device,
    n_repeats=3, top_k=25
)
print(f"Baseline ROC-AUC : {base_auc:.4f}")
print("\nTop 25 features:")
print(imp_df.round(5).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))
colors = ['#e74c3c' if v > 0 else '#95a5a6' for v in imp_df['importance']]
ax.barh(imp_df['feature'][::-1], imp_df['importance'][::-1],
        xerr=imp_df['std'][::-1], color=colors[::-1],
        edgecolor='white', capsize=3, error_kw={'linewidth': 1})
ax.set_xlabel('Mean ROC-AUC Drop When Feature is Permuted', fontsize=12)
ax.set_title('Permutation Feature Importance (Top 25)', fontsize=14)
ax.axvline(0, color='black', linewidth=0.8)
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

## 11. Results Summary & Interpretation

In [ ]:
print("=" * 62)
print("NEURAL NETWORK — FINAL RESULTS SUMMARY")
print("=" * 62)
print(f"Architecture   : MLP {best_config['hidden_dims']} + BatchNorm + Dropout({best_config['dropout_rate']})")
print(f"Parameters     : {final_model.count_parameters():,}")
print(f"Learning rate  : {best_config['lr']}")
print(f"Optimizer      : Adam + ReduceLROnPlateau (factor=0.5, patience=3)")
print(f"Loss           : BCEWithLogitsLoss  pos_weight={pw_tv:.2f}")
print(f"Train+Val rows : {X_tv.shape[0]:,}")
print(f"Test rows      : {X_test.shape[0]:,}")
print()
print("-" * 62)
print("TEST SET METRICS")
print("-" * 62)
print(f"  ROC-AUC             : {test_roc:.4f}")
print(f"  PR-AUC              : {test_pr:.4f}")
print(f"  KS Statistic        : {ks_stat:.4f}  (t={ks_thresh:.3f})")
print(f"  F1 Score (t=0.50)   : {f1_05:.4f}")
print()
print("-" * 62)
print("TOP 5 FEATURES (Permutation Importance)")
print("-" * 62)
for _, row in imp_df.head(5).iterrows():
    print(f"  {row['feature']:<38}  ΔAUC = {row['importance']:.4f} ± {row['std']:.4f}")

summary = {
    'model': 'MLP Neural Network',
    'hidden_dims': str(best_config['hidden_dims']),
    'dropout_rate': best_config['dropout_rate'],
    'lr': best_config['lr'],
    'roc_auc': round(test_roc, 4),
    'pr_auc': round(test_pr, 4),
    'ks_stat': round(ks_stat, 4),
    'f1_at_0.5': round(f1_05, 4),
    'n_params': final_model.count_parameters(),
    'train_val_size': X_tv.shape[0],
    'test_size': X_test.shape[0],
}
pd.DataFrame([summary]).to_csv('models/nn_results.csv', index=False)
print("\n✓ Results saved → models/nn_results.csv")